# 04 - QLoRA training (Qwen2.5-1.5B-Instruct, 4-bit)

GPU required. Same pattern as `03_lora.ipynb`, but the base model is quantized to 4-bit (nf4, double quant) via `bitsandbytes` before training the LoRA adapter on top. Same LoRA hyperparameters as Phase 7, so the only variable that changes is quantization - that's the point of comparison.

In [ ]:
%pip install -q -U torchao transformers datasets accelerate peft trl bitsandbytes pyyaml evaluate rouge_score sentencepiece

In [ ]:
import os
import sys

REPO_URL = "https://github.com/satyazm/Finetuning_LLMs.git"
REPO_DIR = "/kaggle/working/Finetuning_LLMs"

if not os.path.exists(REPO_DIR):
    clone_url = REPO_URL
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
        clone_url = REPO_URL.replace("https://", f"https://{token}@")
    except Exception:
        pass
    !git clone -q {clone_url} {REPO_DIR}

sys.path.append(REPO_DIR)
os.chdir(REPO_DIR)

## Regenerate the dataset

Same as the LoRA notebook: `data/` isn't committed, so pull MedQuAD from the HF Hub again with the same seed/split.

In [ ]:
from src.data.preprocess import run as preprocess_run

preprocess_run(output_dir="data")

## Train

All the QLoRA logic lives in `src/training/train_qlora.py` (same `configs/training.yaml` hyperparameters as LoRA - only the base model's quantization differs). Prints peak training GPU memory when done, for the LoRA-vs-QLoRA memory comparison.

In [ ]:
from src.training.train_qlora import train

trainer = train(output_dir="adapters/qlora")

## Evaluate the QLoRA adapter

Same fixed, seeded 200-example test sample as `02_baseline.ipynb` and `03_lora.ipynb`, so ROUGE/latency are directly comparable across all three.

In [ ]:
import yaml

from src.evaluation.evaluate import run_adapter_eval

with open("configs/model.yaml") as f:
    model_cfg = yaml.safe_load(f)

df_qlora, summary_qlora = run_adapter_eval(
    model_name=model_cfg["base_model"],
    adapter_path="adapters/qlora",
    test_path="data/test.json",
    output_csv="results/qlora.csv",
)

## Verify

In [ ]:
print(trainer.state.log_history[-1])
print(sorted(os.listdir("adapters/qlora")))
print(summary_qlora)